# Coocorrência e matrizes relacionais

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_06/03_coocorrencia_e_matrizes_relacionais.ipynb)

Coocorrência significa que duas entidades satisfazem uma regra de contexto — por
exemplo, aparecem no mesmo documento. Isso não prova contato, colaboração ou
afinidade. A janela e a unidade documental definem a relação.

## 1. Palavras: a janela define o contexto

Em textos, duas palavras podem coocorrer na mesma frase, parágrafo, documento ou
dentro de uma janela de $k$ posições. Janelas menores aproximam contexto local;
documentos inteiros misturam passagens distintas. Remoção de palavras frequentes,
lematização e repetição dentro da janela também alteram as arestas.

| Relação | Nós | Regra de aresta | Direção possível |
|---|---|---|---|
| palavras | termos | mesma janela/frase/documento | normalmente não dirigida |
| entidades | pessoas/instituições | mesma unidade documental | normalmente não dirigida |
| correspondência | pessoas/instituições | remessa registrada | remetente → destinatário |
| citação | obras/autores | referência registrada | citante → citado |
| personagens | personagens | fala, interação ou mesma cena | depende da regra |

![Uma matriz documento-entidade é projetada em uma matriz entidade-entidade, com perda de contexto.](imagens/03_projecao_bipartida.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_06'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
from collections import Counter
from itertools import combinations
import numpy as np
import pandas as pd

tokens = "educação trabalho associação biblioteca educação curso".split()
janela = 2
pares_palavras = Counter()
for i, palavra in enumerate(tokens):
    for vizinha in tokens[i + 1:i + 1 + janela]:
        pares_palavras[tuple(sorted((palavra, vizinha)))] += 1
display(pd.DataFrame(
    [(a, b, peso) for (a, b), peso in pares_palavras.items()],
    columns=["palavra_a", "palavra_b", "peso"],
).sort_values("peso", ascending=False))

# Agora mudamos de palavras para entidades por documento.
documentos = pd.read_csv("dados/documentos_entidades.csv")
mencoes = documentos.assign(entidade=documentos["entidades"].str.split(";")).explode("entidade")
incidencia = pd.crosstab(mencoes["id_documento"], mencoes["entidade"]).astype(int)
display(mencoes.head(8))
display(incidencia.iloc[:6, :7])

## 2. Projetar documentos em pares de entidades

Se $B$ é a matriz documento-entidade, $B^TB$ conta quantos documentos cada par
compartilha. A diagonal conta documentos por entidade e deve ser separada quando
o objetivo são arestas entre entidades.

$$W=B^TB$$

In [ ]:
matriz_coocorrencia = incidencia.T @ incidencia
np.fill_diagonal(matriz_coocorrencia.values, 0)
entidades = matriz_coocorrencia.index.tolist()
arestas = pd.DataFrame(
    [
        (entidades[i], entidades[j], int(matriz_coocorrencia.iat[i, j]))
        for i in range(len(entidades))
        for j in range(i + 1, len(entidades))
        if matriz_coocorrencia.iat[i, j] > 0
    ],
    columns=["origem", "destino", "peso"],
).sort_values(["peso", "origem"], ascending=[False, True]).reset_index(drop=True)
display(arestas.head(12))

A projeção simplifica uma **rede bipartida**: depois dela, nem sempre sabemos
quais documentos produziram cada peso. Preserve a tabela de menções e retorne aos
registros. Documentos com muitas entidades também geram muitos pares e podem
dominar a rede.

## 3. Coocorrência não é correspondência

`correspondencia_direta.csv` contém remetente e destinatário: é uma relação
dirigida observada no modelo dos dados. A coocorrência é simétrica e construída
pelo analista. As duas redes não são intercambiáveis.

## U06-A04 — Atividade integrada — matriz relacional auditável

Altere o recorte por período ou local, reconstrua incidência, matriz e arestas,
compare os cinco pares mais fortes e recupere os documentos que sustentam um par.
Explique o que “peso” significa e o que não significa.

**Resposta:** Escreva aqui.

## Passagem

Temos agora uma lista de nós e arestas com proveniência. O Notebook 04 usa o
grafo como modelo analítico, sem transformar posição estrutural em importância
histórica automática.